# 01 · Scan Quality Check — is this tile in focus?

**Problem.** Out-of-focus regions are the most common reason a whole-slide scan is rejected. Modern scanners run
*built-in quality assessment* so a bad scan is flagged (and rescanned) **before** it reaches a pathologist.

**What this notebook does**
1. Measures how classical autofocus metrics respond to optical defocus.
2. Shows their weakness: scores depend on *tissue content*, not only blur → no absolute threshold.
3. Trains **BlurNet**, a tiny CNN (~17k params) that predicts an **absolute defocus level**, using *synthetic* defocus
   (disk point-spread function) on PatchCamelyon H&E patches.
4. Tests on **real scanner blur** (FocusPath: 9 stains × 14 z-levels) — if downloaded.
5. Produces a slide-level QC heatmap and exports BlurNet to ONNX for edge inference.

In [ ]:
# --- setup: works locally (run from repo root or notebooks/) and on Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("wsi-smart-scan"):
    !git clone https://github.com/DPK516/wsi-smart-scan.git
    %cd wsi-smart-scan
    !pip -q install -r requirements.txt
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
os.makedirs("results", exist_ok=True); os.makedirs("models", exist_ok=True); os.makedirs("assets", exist_ok=True)

In [ ]:
from skimage import data
from src.quality import FOCUS_METRICS, apply_defocus, focus_map, relative_blur, qc_decision
from src.utils import to_uint8, overlay_heatmap, tissue_mask

QUICK = False          # True = tiny run for testing the notebook end-to-end
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1 · Classical focus metrics vs. defocus radius

In [ ]:
tissue = data.immunohistochemistry()                       # bundled sample (no download needed)
radii = np.arange(0, 6.5, 0.5)
curves = {m: [] for m in FOCUS_METRICS}
for r in radii:
    blurred = to_uint8(apply_defocus(tissue, r))
    for m, fn in FOCUS_METRICS.items():
        curves[m].append(fn(blurred))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for m, v in curves.items():
    ax[0].plot(radii, np.array(v) / v[0], marker="o", label=m)
ax[0].set(xlabel="defocus radius (px)", ylabel="score / score at focus", title="Metric response to defocus")
ax[0].legend()
ax[1].imshow(np.hstack([tissue[:200, :200], to_uint8(apply_defocus(tissue[:200, :200], 2)),
                        to_uint8(apply_defocus(tissue[:200, :200], 5))]))
ax[1].set_title("radius 0  |  2 px  |  5 px"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 2 · The weakness: classical scores also depend on tissue content

A sharpness score mixes *focus* with *how much texture the tile has* (dense nuclei vs. pale stroma vs. tissue edge).
We measure how much sharp-tile scores vary, and how often a perfectly sharp tile scores below a mildly blurred one -
the cases where a fixed threshold fails. This motivates a learned, content-aware model.

In [ ]:
from src.pipeline import demo_slide
slide = demo_slide(seed=3); mask = tissue_mask(slide); T = 64
tiles = [slide[y:y+T, x:x+T] for y in range(0, slide.shape[0], T) for x in range(0, slide.shape[1], T)
         if mask[y:y+T, x:x+T].mean() > 0.3]
lv = FOCUS_METRICS["laplacian_var"]
sharp = np.array([lv(t) for t in tiles])
print(f"{len(tiles)} tissue tiles | sharp-tile score varies {sharp.max() / sharp.min():.1f}x across tissue content")
rows = []
for r in [0.5, 1.0, 1.5, 2.0, 3.0]:
    blurred = np.array([lv(to_uint8(apply_defocus(t, r))) for t in tiles])
    rows.append({"blur radius (px)": r, "sharp tiles scoring below sharpest blurred tile": (sharp < blurred.max()).mean()})
pd.DataFrame(rows).round(3)

## 3 · Train BlurNet on synthetic defocus

Data: PatchCamelyon H&E patches (96×96, ~1 µm/px). Each patch is blurred on-the-fly with a random disk PSF radius
in [0, 6] px plus sensor noise and illumination jitter; the network regresses the radius.
Run `python data/download_data.py` first.

In [ ]:
import h5py, torch
from torch.utils.data import DataLoader
from src.quality import BlurNet, SyntheticDefocusDataset, train_blurnet, predict_loader, export_blurnet_onnx
torch.manual_seed(0)

n_train, n_val = (512, 128) if QUICK else (20000, 4000)
with h5py.File("data/pcam/camelyonpatch_level_2_split_valid_x.h5") as f:
    train_patches = f["x"][:n_train]
with h5py.File("data/pcam/camelyonpatch_level_2_split_test_x.h5") as f:
    val_patches = f["x"][:n_val]                       # different split = unseen tissue
print(train_patches.shape, val_patches.shape)

train_dl = DataLoader(SyntheticDefocusDataset(train_patches), batch_size=128, shuffle=True, num_workers=2)
val_dl = DataLoader(SyntheticDefocusDataset(val_patches, seed=1), batch_size=256, num_workers=2)
model = BlurNet()
print("parameters:", sum(p.numel() for p in model.parameters()))
history = train_blurnet(model, train_dl, val_dl, epochs=1 if QUICK else 15, lr=3e-3, device=DEVICE)
torch.save(model.state_dict(), "models/blurnet.pt")

## 4 · Held-out synthetic test: BlurNet vs classical metrics

In [ ]:
from scipy.stats import spearmanr, pearsonr
# build ONE fixed blurred test set, then score it with every method
test_ds = SyntheticDefocusDataset(val_patches, seed=7)
X, Y = zip(*[test_ds[i] for i in range(len(test_ds))])
X, true = torch.stack(X), torch.stack(Y).numpy()
model.eval()
with torch.no_grad():
    pred = torch.cat([model(X[i:i+256].to(DEVICE)).cpu() for i in range(0, len(X), 256)]).numpy()
imgs = [to_uint8(np.clip(x.numpy().transpose(1, 2, 0), 0, 1)) for x in X]

res = {"BlurNet": {"SRCC": spearmanr(pred, true).correlation, "MAE_px": float(np.abs(pred - true).mean())}}
for m, fn in FOCUS_METRICS.items():   # sharper = higher score -> negate to correlate with blur
    s = np.array([fn(im) for im in imgs])
    res[m] = {"SRCC": spearmanr(-s, true).correlation, "MAE_px": np.nan}
synthetic_table = pd.DataFrame(res).T.round(3)
synthetic_table

In [ ]:
plt.figure(figsize=(4.5, 4))
plt.scatter(true, pred, s=4, alpha=0.3)
plt.plot([0, 6], [0, 6], "r--"); plt.xlabel("true defocus radius (px)"); plt.ylabel("BlurNet prediction")
plt.title("BlurNet on unseen tissue"); plt.tight_layout(); plt.savefig("assets/blurnet_scatter.png", dpi=120); plt.show()

## 5 · Real scanner blur — FocusPath (optional, 13 GB)

`python data/download_data.py --skip-pcam --focuspath`. FocusPath images are 40× (0.25 µm/px); we downsample 4× to
~1 µm/px so the pixel scale matches the PCam training data. Ground truth = |z-level| (distance from focus).
**This is the real test**: the model never saw real optical blur during training.

In [ ]:
import glob
fp_csv = "data/focuspath/FocusPath_full_split1.txt"
fp_root = next(iter(glob.glob("data/focuspath/**/FocusPath_full", recursive=True)), None)
focuspath_table = None
if fp_root and os.path.exists(fp_csv):
    from src.quality import FocusPathDataset
    fp = FocusPathDataset(fp_csv, fp_root, limit=300 if QUICK else 2000)
    fp_pred, fp_true = predict_loader(model, DataLoader(fp, batch_size=16, num_workers=2), DEVICE)
    rows = {"BlurNet": {"SRCC": spearmanr(fp_pred, fp_true).correlation, "PLCC": pearsonr(fp_pred, fp_true)[0]}}
    imgs = [fp.load(i) for i in range(len(fp))]
    for m, fn in FOCUS_METRICS.items():
        s = -np.array([fn(im) for im in imgs])
        rows[m] = {"SRCC": spearmanr(s, fp_true).correlation, "PLCC": pearsonr(s, fp_true)[0]}
    focuspath_table = pd.DataFrame(rows).T.round(3)
    # per-stain robustness: correlation within each of the 9 slides
    per_slide = pd.DataFrame({"slide": fp.slide, "pred": fp_pred, "z": fp_true}).groupby("slide").apply(
        lambda g: spearmanr(g.pred, g.z).correlation).rename("BlurNet SRCC")
    display(focuspath_table); display(per_slide.round(3))
else:
    print("FocusPath not found - skipping real-blur evaluation.")

## 6 · Slide-level QC heatmap

In [ ]:
from src.pipeline import demo_slide
from src.quality import OnnxBlurScorer
export_blurnet_onnx(model, "models/blurnet.onnx")
scorer = OnnxBlurScorer("models/blurnet.onnx")

slide = demo_slide(seed=3).copy()
slide[300:460, 380:560] = to_uint8(apply_defocus(slide[300:460, 380:560], 4))    # inject an out-of-focus region
mask, T = tissue_mask(slide), 64
grid = np.full((slide.shape[0] // T, slide.shape[1] // T), np.nan)
for r in range(grid.shape[0]):
    for c in range(grid.shape[1]):
        if mask[r*T:(r+1)*T, c*T:(c+1)*T].mean() > 0.3:
            grid[r, c] = scorer([slide[r*T:(r+1)*T, c*T:(c+1)*T]])[0]
qc = qc_decision(grid, threshold=2.0)
print("verdict:", qc.status, "| failed tiles:", len(qc.bad_tiles))
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].imshow(slide); ax[0].set_title("Slide with injected defocus")
ax[1].imshow(overlay_heatmap(slide, grid, T, cmap="RdYlGn_r", vmin=0, vmax=5)); ax[1].set_title("BlurNet defocus (px)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig("assets/blurnet_qc_heatmap.png", dpi=120); plt.show()

## 7 · Save results

In [ ]:
out = {"synthetic_test": synthetic_table.to_dict(),
       "focuspath": None if focuspath_table is None else focuspath_table.to_dict(),
       "blurnet_params": sum(p.numel() for p in model.parameters()),
       "training_history": history}
json.dump(out, open("results/quality_results.json", "w"), indent=2, default=float)
print("saved results/quality_results.json and models/blurnet.onnx")